# So sánh chất lượng giữa các lần thử nghiệm

Notebook này **đọc** báo cáo đã ghi sẵn trên đĩa, không huấn luyện lại. Chạy trước (ít nhất một lần,
không cần đủ cả mười biến thể):

```bash
uv run python -m app.scripts.train_risk_model_experiments
```

Nguồn dữ liệu: `evaluation_metrics.csv` và `evaluation_report.json` của mô hình sản xuất
(`RISK_MODEL_DIR`) và của mọi thư mục con trong `experiments/` mà `train_risk_model_experiments.py`
đã ghi. Ghép lại thành bảng và biểu đồ so sánh Accuracy/F1/ROC-AUC theo (biến thể, thuật toán, mốc dự
đoán).

**Lưu notebook với ô kết quả đã xoá sạch** — cùng quy ước với `risk_model_analysis.ipynb`: số liệu đã
nằm ở JSON/CSV, giữ thêm bản trong notebook chỉ làm git diff nhiễu.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from app.core.config import settings
from app.risk.predictor import CHECKPOINTS
from app.risk.training import METRICS_FILENAME, REPORT_FILENAME

BASELINE_DIR = Path(settings.RISK_MODEL_DIR)
EXPERIMENTS_DIR = BASELINE_DIR.parent / "experiments"  # sibling của RISK_MODEL_DIR, khớp #45

variant_dirs = sorted(
    path for path in EXPERIMENTS_DIR.glob("*") if (path / METRICS_FILENAME).exists()
)

print(f"Baseline (sản xuất): {BASELINE_DIR}")
print(f"Tìm thấy {len(variant_dirs)} biến thể đã có báo cáo trong {EXPERIMENTS_DIR}:")
for path in variant_dirs:
    print(f"  - {path.name}")

## 1. Bảng so sánh Accuracy / F1 / ROC-AUC

`roc_auc` để trống ở biến thể/thuật toán/mốc nào tập kiểm tra chỉ còn một lớp nhãn (#41) — không
phải lỗi đọc tệp. `accuracy` và `f1` đo tại ngưỡng đề xuất riêng của từng biến thể (mỗi biến thể có
tập kiểm định, và do đó ngưỡng, khác nhau); `roc_auc` không phụ thuộc ngưỡng nào — ba cột không cùng
"thước đo" nên đừng cộng dồn hay lấy trung bình chúng với nhau.

In [ ]:
def read_metrics(directory: Path, label: str) -> pd.DataFrame:
    frame = pd.read_csv(directory / METRICS_FILENAME)
    frame.insert(0, "biến thể", label)
    return frame[["biến thể", "algorithm", "checkpoint", "accuracy", "f1", "roc_auc"]]


frames = [read_metrics(BASELINE_DIR, "baseline (sản xuất)")]
frames += [read_metrics(path, path.name) for path in variant_dirs]

metrics = pd.concat(frames, ignore_index=True).rename(
    columns={"algorithm": "thuật toán", "checkpoint": "mốc"}
)

metrics.pivot_table(
    index=["biến thể", "thuật toán"], columns="mốc", values=["accuracy", "f1", "roc_auc"]
)

## 2. Ngữ cảnh mỗi biến thể

Thuật toán nào được chọn, ngưỡng đề xuất bao nhiêu, và biến thể đó có đạt mục tiêu F1 >= 0,30 ở mốc
đặt hàng hay không — cùng tiêu chí chọn thuật toán với lệnh huấn luyện sản xuất (ADR-0008), áp dụng
riêng cho từng biến thể.

In [ ]:
def read_context(directory: Path, label: str) -> dict:
    report = json.loads((directory / REPORT_FILENAME).read_text("utf-8"))
    return {
        "biến thể": label,
        "thuật toán được chọn": report["selected_algorithm"],
        "ngưỡng đề xuất": report["suggested_risk_threshold"],
        "F1 ở mốc đặt hàng": report["f1_at_order_placed"],
        "đạt mục tiêu F1": report["meets_f1_target"],
    }


context = pd.DataFrame(
    [read_context(BASELINE_DIR, "baseline (sản xuất)")]
    + [read_context(path, path.name) for path in variant_dirs]
)
context

## 3. Biểu đồ so sánh theo mốc dự đoán

Mỗi thanh là một (biến thể, thuật toán), sắp theo giá trị chỉ số, một biểu đồ con cho mỗi mốc dự
đoán. Biến thể nào chỉ thử một thuật toán (ví dụ các biến thể LightGBM) chỉ có một thanh.

In [ ]:
metrics["nhãn"] = metrics["biến thể"] + " / " + metrics["thuật toán"]

for column in ("accuracy", "f1", "roc_auc"):
    height = 0.35 * metrics["nhãn"].nunique() + 1
    fig, axes = plt.subplots(1, len(CHECKPOINTS), figsize=(6 * len(CHECKPOINTS), height), sharey=True)

    for axis, checkpoint in zip(axes, CHECKPOINTS):
        part = metrics[metrics["mốc"] == checkpoint].dropna(subset=[column]).sort_values(column)
        axis.barh(part["nhãn"], part[column])
        axis.set_title(checkpoint)
        axis.set_xlabel(column)

    fig.suptitle(f"So sánh {column} theo biến thể / thuật toán")
    plt.tight_layout()
    plt.show()